# Vector Search Concepts

## From keyword matching to semantic understanding

A historian types "medieval trade routes" into the search box.

Keyword search returns nothing. The texts in our collection use the words "commerce" and "Middle Ages", not "trade routes" and "medieval". The knowledge is there. The search just cannot find it.

This is the fundamental limitation of keyword search: it matches words, not meaning. Two texts about exactly the same topic, written with different vocabulary, will not match. A researcher has to guess which words the author used — and if they guess wrong, they get nothing.

Vector search solves this problem. Instead of matching strings, it matches *meaning*. Texts are converted into numerical vectors — points in a high-dimensional space — where similar meanings cluster together. "Medieval trade routes" and "commerce in the Middle Ages" end up as nearby points, and the search finds them both.

In this notebook, we will build three search approaches from scratch and compare them on the same queries.

In [ ]:
%pip install -q scikit-learn pyarrow

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Loading pre-computed data

We have roughly 500 text chunks, each from the British Library collection, with pre-computed embeddings (384-dimensional TF-IDF vectors). In a production system, these would be dense embeddings from a model like `all-MiniLM-L6-v2`, but TF-IDF vectors work well enough to demonstrate the concepts — and they run entirely in the browser.

In [ ]:
# Load chunks with embeddings
chunks_df = pd.read_parquet("../data/bl_chunks_with_embeddings.parquet")
embeddings = np.load("../data/bl_embeddings.npy")

print(f"Chunks: {len(chunks_df)}")
print(f"Embeddings shape: {embeddings.shape}")
print(f"\nColumns: {list(chunks_df.columns)}")
chunks_df.head()

In [ ]:
# Peek at the chunk texts
for i in [0, 100, 250, 400]:
    print(f"--- Chunk {chunks_df.iloc[i]['chunk_id']} ---")
    print(chunks_df.iloc[i]["chunk_text"][:200])
    print()

## Approach 1: Keyword search

The simplest search: check whether the query string (or any of its words) appears in the text. This is essentially what `str.contains()` does.

In [ ]:
def keyword_search(query, chunks_df, top_k=5):
    """Simple keyword search: count query term occurrences in each chunk."""
    query_terms = query.lower().split()
    
    scores = []
    for text in chunks_df["chunk_text"]:
        text_lower = text.lower()
        score = sum(text_lower.count(term) for term in query_terms)
        scores.append(score)
    
    chunks_df = chunks_df.copy()
    chunks_df["score"] = scores
    results = chunks_df[chunks_df["score"] > 0].nlargest(top_k, "score")
    
    return results[["chunk_id", "score", "chunk_text"]]

In [ ]:
# Test keyword search
query = "medieval trade routes"
results = keyword_search(query, chunks_df)

print(f'Keyword search for "{query}":')
print(f"Results found: {len(results)}")

if len(results) > 0:
    for _, row in results.iterrows():
        print(f"\n  Score: {row['score']} | {row['chunk_id']}")
        print(f"  {row['chunk_text'][:150]}...")
else:
    print("  No results. The texts use different words for the same concepts.")

Keyword search looks for the exact words "medieval", "trade", and "routes". If the texts use "commerce", "Middle Ages", or "wool trade" instead, keyword search misses them entirely. This is the vocabulary mismatch problem, and it is the reason we need something better.

## Approach 2: TF-IDF search

TF-IDF (Term Frequency-Inverse Document Frequency) is a step up from keyword matching. Instead of treating all words equally, it gives more weight to words that are distinctive.

The intuition:
- **Term Frequency (TF)**: How often does this word appear in *this* document? A word that appears five times in a chunk is probably more relevant to that chunk than a word that appears once.
- **Inverse Document Frequency (IDF)**: How common is this word across *all* documents? Words like "the" and "and" appear everywhere and tell us nothing. Words like "feudal" or "cholera" are rare and distinctive.

TF-IDF = TF x IDF. A high TF-IDF score means the word is frequent in this document but rare overall — exactly the kind of word that makes a document distinctive.

In [ ]:
# Build a TF-IDF index over all chunks
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
tfidf_matrix = tfidf.fit_transform(chunks_df["chunk_text"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"\nSome vocabulary terms: {list(tfidf.vocabulary_.keys())[:20]}")

In [ ]:
def tfidf_search(query, tfidf_vectorizer, tfidf_matrix, chunks_df, top_k=5):
    """Search using TF-IDF cosine similarity."""
    query_vec = tfidf_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    
    chunks_df = chunks_df.copy()
    chunks_df["score"] = similarities
    results = chunks_df.nlargest(top_k, "score")
    
    return results[["chunk_id", "score", "chunk_text"]]

In [ ]:
# Test TF-IDF search
results = tfidf_search(query, tfidf, tfidf_matrix, chunks_df)

print(f'TF-IDF search for "{query}":')
for _, row in results.iterrows():
    print(f"\n  Score: {row['score']:.4f} | {row['chunk_id']}")
    print(f"  {row['chunk_text'][:150]}...")

TF-IDF does better. It picks up on shared vocabulary — if the query mentions "medieval" and a document mentions "medieval", the IDF weighting recognises that as a meaningful match because "medieval" is a relatively rare word. But TF-IDF still relies on exact word overlap. "Commerce" and "trade" do not match, even though they mean the same thing.

## Approach 3: Vector (embedding) search

Vector search represents each piece of text as a point in a high-dimensional space. The key property: texts with similar *meaning* end up as nearby points, regardless of whether they share any specific words.

How does this work? A text embedding model (a neural network trained on enormous amounts of text) learns to map sentences and paragraphs into vectors such that:
- "medieval trade routes" is near "commerce in the Middle Ages"
- "the Black Death" is near "bubonic plague in fourteenth-century England"
- Both are far from "Victorian railway engineering"

We cannot run an actual embedding model in the browser, but our pre-computed TF-IDF embeddings capture some of this semantic signal. Let us use them.

### Cosine similarity from scratch

Before we use sklearn's implementation, let us understand what cosine similarity actually computes.

Given two vectors **a** and **b**, the cosine similarity is:

$$\text{cosine\_similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \times \|\mathbf{b}\|}$$

The numerator is the dot product: element-wise multiplication, then sum. The denominator normalises by the lengths (magnitudes) of both vectors.

The result is always between -1 and 1 (or 0 and 1 for non-negative vectors like TF-IDF):
- 1.0 means identical direction (identical meaning)
- 0.0 means orthogonal (unrelated)
- -1.0 means opposite direction (opposite meaning — rare in practice)

In [ ]:
def cosine_sim(a, b):
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return dot_product / (norm_a * norm_b)

# Test: similarity between identical vectors
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([1.0, 2.0, 3.0])
print(f"Identical vectors: {cosine_sim(v1, v2):.4f}")

# Test: orthogonal vectors
v3 = np.array([1.0, 0.0, 0.0])
v4 = np.array([0.0, 1.0, 0.0])
print(f"Orthogonal vectors: {cosine_sim(v3, v4):.4f}")

# Test: similar but not identical
v5 = np.array([1.0, 2.0, 3.0])
v6 = np.array([1.1, 2.1, 2.9])
print(f"Similar vectors: {cosine_sim(v5, v6):.4f}")

Now let us use our pre-computed embeddings to search. We need to build a TF-IDF vectoriser on the chunk texts (matching how the embeddings were generated), then transform the query into the same vector space.

In [ ]:
# Build a TF-IDF vectoriser matching the embedding dimensions (384)
embed_tfidf = TfidfVectorizer(max_features=384)
embed_matrix = embed_tfidf.fit_transform(chunks_df["chunk_text"]).toarray().astype(np.float32)

# Normalise to unit vectors (so dot product = cosine similarity)
norms = np.linalg.norm(embed_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1
embed_matrix = embed_matrix / norms

print(f"Embedding matrix shape: {embed_matrix.shape}")

In [ ]:
def vector_search(query, vectorizer, embedding_matrix, chunks_df, top_k=5):
    """Search using pre-computed vector embeddings."""
    # Transform query into the same vector space
    query_vec = vectorizer.transform([query]).toarray().astype(np.float32)
    query_norm = np.linalg.norm(query_vec)
    if query_norm > 0:
        query_vec = query_vec / query_norm
    
    # Compute cosine similarity with all chunks
    similarities = cosine_similarity(query_vec, embedding_matrix)[0]
    
    chunks_df = chunks_df.copy()
    chunks_df["score"] = similarities
    results = chunks_df.nlargest(top_k, "score")
    
    return results[["chunk_id", "score", "chunk_text"]]

In [ ]:
# Test vector search
results = vector_search(query, embed_tfidf, embed_matrix, chunks_df)

print(f'Vector search for "{query}":')
for _, row in results.iterrows():
    print(f"\n  Score: {row['score']:.4f} | {row['chunk_id']}")
    print(f"  {row['chunk_text'][:150]}...")

## The comparison: three approaches, one query

Let us run all three approaches side by side on several queries and see the difference.

In [ ]:
def compare_search(query, top_k=3):
    """Run all three search approaches and compare results."""
    print(f'\n{"=" * 70}')
    print(f'Query: "{query}"')
    print(f'{"=" * 70}')
    
    # Keyword search
    kw_results = keyword_search(query, chunks_df, top_k=top_k)
    print(f"\n--- Keyword search: {len(kw_results)} results ---")
    if len(kw_results) == 0:
        print("  No results found.")
    for _, row in kw_results.iterrows():
        print(f"  [{row['score']:.0f}] {row['chunk_text'][:100]}...")
    
    # TF-IDF search
    tfidf_results = tfidf_search(query, tfidf, tfidf_matrix, chunks_df, top_k=top_k)
    print(f"\n--- TF-IDF search ---")
    for _, row in tfidf_results.iterrows():
        print(f"  [{row['score']:.4f}] {row['chunk_text'][:100]}...")
    
    # Vector search
    vec_results = vector_search(query, embed_tfidf, embed_matrix, chunks_df, top_k=top_k)
    print(f"\n--- Vector search ---")
    for _, row in vec_results.iterrows():
        print(f"  [{row['score']:.4f}] {row['chunk_text'][:100]}...")

In [ ]:
compare_search("medieval trade routes")

In [ ]:
compare_search("impact of plague on workers")

In [ ]:
compare_search("early computers and codebreaking")

In [ ]:
compare_search("women in science")

In [ ]:
compare_search("how did factories change daily life")

## Nearest neighbour search

Vector search is fundamentally a **nearest neighbour** problem: given a query point, find the k closest points in the embedding space. Let us implement this explicitly.

In [ ]:
def find_nearest(query, vectorizer, embedding_matrix, chunks_df, k=5):
    """Find the k nearest chunks to a query in embedding space."""
    # Embed the query
    query_vec = vectorizer.transform([query]).toarray().astype(np.float32)
    query_norm = np.linalg.norm(query_vec)
    if query_norm > 0:
        query_vec = query_vec / query_norm
    
    # Compute all distances (1 - cosine similarity = cosine distance)
    similarities = cosine_similarity(query_vec, embedding_matrix)[0]
    distances = 1 - similarities
    
    # Find k smallest distances
    nearest_idx = np.argsort(distances)[:k]
    
    results = []
    for idx in nearest_idx:
        results.append({
            "rank": len(results) + 1,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "similarity": similarities[idx],
            "distance": distances[idx],
            "text_preview": chunks_df.iloc[idx]["chunk_text"][:120] + "...",
        })
    
    return pd.DataFrame(results)

In [ ]:
# Find the 5 nearest chunks to a query
nearest = find_nearest("sewage and public health in Victorian London", 
                       embed_tfidf, embed_matrix, chunks_df, k=5)
nearest

## Understanding embeddings geometrically

Embeddings are vectors in a 384-dimensional space. We cannot visualise 384 dimensions, but we can examine the relationships between vectors numerically.

In [ ]:
# Pick three chunks on different topics
medieval_idx = None
victorian_idx = None
computing_idx = None

for i, text in enumerate(chunks_df["chunk_text"]):
    text_lower = text.lower()
    if "feudal" in text_lower and medieval_idx is None:
        medieval_idx = i
    elif "cholera" in text_lower and victorian_idx is None:
        victorian_idx = i
    elif "turing" in text_lower and computing_idx is None:
        computing_idx = i
    if all(x is not None for x in [medieval_idx, victorian_idx, computing_idx]):
        break

if all(x is not None for x in [medieval_idx, victorian_idx, computing_idx]):
    print("Three chunks from different eras:\n")
    for label, idx in [("Medieval", medieval_idx), ("Victorian", victorian_idx), ("Computing", computing_idx)]:
        print(f"  {label}: {chunks_df.iloc[idx]['chunk_text'][:80]}...")
    
    print("\nPairwise cosine similarities:")
    pairs = [
        ("Medieval vs Victorian", medieval_idx, victorian_idx),
        ("Medieval vs Computing", medieval_idx, computing_idx),
        ("Victorian vs Computing", victorian_idx, computing_idx),
    ]
    for label, i, j in pairs:
        sim = cosine_sim(embed_matrix[i], embed_matrix[j])
        print(f"  {label}: {sim:.4f}")

The similarities reflect topical relatedness. Chunks about different historical periods are more related to each other than to a chunk about computing, because they share vocabulary about England, history, and society.

With dense embeddings from a neural model, these relationships would be even more nuanced — the model would understand that "the Black Death" and "cholera" are both epidemic diseases even though they share no words.

## From TF-IDF to dense embeddings

Our TF-IDF vectors are a useful bridge, but they have a fundamental limitation: they can only capture similarity through shared vocabulary. Two documents about the same topic that use entirely different words will have zero TF-IDF similarity.

Dense embedding models (like Sentence-BERT, OpenAI's embedding API, or Cohere's embed endpoint) solve this by training on billions of text pairs. They learn that:
- "pupil" and "student" are interchangeable
- "the monarch" and "the king or queen" mean the same thing
- "rise in wages" is related to "labour shortage" even though no words overlap

These models produce fixed-size vectors (typically 384 or 768 dimensions) that encode semantic meaning. The vector space is continuous — you can interpolate between concepts, find analogies, and cluster by topic.

We cannot run these models in the browser (they are too large for WASM), but in a production pipeline you would:
1. Send each chunk to an embedding API
2. Store the returned vectors alongside the chunk text
3. At search time, embed the query with the same model
4. Find the nearest vectors using cosine similarity

The data engineering work — cleaning, chunking, storing, indexing — is identical. Only the embedding step changes.

---

## Exercises

### Exercise 1: Query comparison

Run the `compare_search` function on at least five queries of your own choosing, each related to a different topic in the collection (medieval history, Tudor exploration, the Industrial Revolution, Victorian public health, early computing).

For each query, note:
1. How many keyword results were returned?
2. What was the top TF-IDF score?
3. What was the top vector search score?
4. Which approach found the most relevant result?

Present your findings in a table.

In [ ]:
# Your code here


### Exercise 2: Implement cosine similarity without sklearn

Write a function `batch_cosine_similarity(query_vec, matrix)` that takes a single query vector and a matrix of document vectors and returns an array of cosine similarities. Use only numpy — no sklearn.

Verify that your implementation gives the same results as `sklearn.metrics.pairwise.cosine_similarity` on a sample query.

In [ ]:
# Your code here


### Exercise 3: Similarity threshold

Instead of returning the top-k results, return all results above a similarity threshold. Try thresholds of 0.1, 0.2, and 0.3 on the query "medieval agriculture and labour".

How does the number of results change with the threshold? Is there a natural gap between "relevant" and "irrelevant" results?

In [ ]:
# Your code here


### Your turn: Build a search interface

Write a function `search(query, method="vector", top_k=5)` that:
1. Accepts a query string and a method ("keyword", "tfidf", or "vector")
2. Returns a DataFrame with columns: `rank`, `chunk_id`, `score`, `text_preview`
3. Prints a nicely formatted summary of the results

Test it with several queries.

In [ ]:
# Your code here


---

## Summary

In this notebook we built and compared three search approaches:

1. **Keyword search** — matches exact words. Fast and simple, but fails when query and document use different vocabulary.
2. **TF-IDF search** — weights words by distinctiveness. Better than keywords, but still relies on word overlap.
3. **Vector search** — represents texts as points in a continuous space where similar meanings cluster together. Handles vocabulary mismatch.

We also covered:
- **Cosine similarity** — the standard measure of vector similarity, computed as the normalised dot product
- **Nearest neighbour search** — finding the k closest vectors to a query
- **The gap between TF-IDF and dense embeddings** — TF-IDF gives us real semantic signal, but dense models capture much richer meaning

The big idea: "medieval trade routes" and "commerce in the Middle Ages" mean the same thing. Keyword search does not know this. Vector search does. The entire purpose of the embedding step in a modern search pipeline is to bridge the gap between words and meaning.

Next up: we wire everything together into a Retrieval-Augmented Generation pipeline — the architecture behind every modern AI search system.